## Code mới

In [ ]:
from langchain_core.messages import BaseMessage, ToolMessage, HumanMessage, AIMessage
from langchain.schema.output_parser import StrOutputParser
from typing import Sequence
from typing_extensions import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import START, StateGraph, END
from IPython.display import Image, display
import json
from langchain.prompts import PromptTemplate, HumanMessagePromptTemplate, ChatPromptTemplate, MessagesPlaceholder
import sys
sys.path.append(".")
from configs.config import Load_config
CONFIG = Load_config()
from models.models import Groq_loader
from app.tools.text_to_sql import text_to_sql
from app.tools.other_tools import *
from app.prompt.prompt_template import *

##############################################################

# input format of model
class DictState(TypedDict):
    # messages: Annotated[Sequence[BaseMessage], add_messages]
    messages: Annotated[list, add_messages]
    name: str
    birthday: str

def to_dict(query: str) -> dict:
    return { "messages": [HumanMessage(query)]}

agent_chatprompt = ChatPromptTemplate.from_messages(
    [
        ("system", AGENT_SYSTEM),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

##############################################################

model_loader = Groq_loader()
llm_model = model_loader.create_model()
tools = [search, human_birthday]

class LLM_agent:
    def __init__(self,llm_model, tools, user_infor, session_id):
        self.model_name = user_infor
        self.session_id = session_id

        ## Create agent
        agent = llm_model.bind_tools(tools)
        self.runable = agent_chatprompt | agent

        builder = StateGraph(state_schema=DictState)
        builder.add_edge(START, "chatbot")
        builder.add_node("chatbot", self.call_agent)
        tool_node = ToolNode(tools)
        builder.add_node("mytools", tool_node)
        builder.add_conditional_edges(
             "chatbot", tools_condition, {"tools": "mytools", END: END}
        )
        builder.add_edge("mytools", "chatbot") # return to chatbot to decide next step

        memory = MemorySaver()
        self.graph = builder.compile(checkpointer=memory)
        self.configs = {"configurable": {"thread_id": self.session_id}}

    def call_agent(self,state: DictState):  
        # query = state["messages"][-1].content
        # print(query)
        response = self.runable.invoke(state) # input: list([Basemessage])
        print("Tool calls:", response.tool_calls)  # Debug
        assert len(response.tool_calls) <= 1
        return {"messages": [response]} # Update message history with response
    
    # Replace by tools_condition
    def route_tools(self, state: DictState):
        # get last message
        if isinstance(state, list): # if state is a list
            ai_message = state[-1]
        elif messages := state.get("messages", []): # if state is a dict
            ai_message = messages[-1]
        else:
            raise ValueError(f"No messages found in input state to tool_edge: {state}")
        if hasattr(ai_message, "tool_calls") and len(ai_message.tool_calls) > 0: #
            return "tools"
        return END
    
    def display(self):
        img = Image(self.graph.get_graph().draw_mermaid_png())
        display(img)
        with open("langgraph_architect.png", "wb") as f:
            f.write(img.data)
    
    def print_history(self):
        state = self.graph.get_state(self.configs).values
        for message in state["messages"]:
            message.pretty_print()

    def stream_run(self,query):
        input_dict = to_dict(query)
        try:
            for events in self.graph.stream(input_dict,self.configs):
                for event in events:
                    print("Assistant: ", event["messages"][-1].content)
        except Exception as e:
            print(e)

    def run(self, query):
        input_dict = to_dict(query)
        try:
            output = self.graph.invoke(input_dict,self.configs)
            response = output["messages"][-1].content # last message = last AI response
            print(response)
            return response
        except Exception as e:
            print(e)

    def human_command(self, human_response):
        human_response = Command(resume={"data": human_response})
        output = self.graph.invoke(human_response,self.configs)
        return output["messages"][-1].content

## Replace by ToolNode
class BasicToolNode:
    """A node that runs the tools requested in the last AIMessage."""

    def __init__(self, tools: list) -> None:
        self.tools_dict = {tool.name: tool for tool in tools} # dict of tools

    def __call__(self, inputs: dict):
        if messages := inputs.get("messages", []):
            message = messages[-1]  # get last message
        else:
            raise ValueError("No message found in input")
        
        outputs = []
        for tool in message.tool_calls: # list of tool agent decide to call.
            tool_result = self.tools_dict[tool["name"]].invoke(
                tool["args"] # variable send to tool
            )
            outputs.append(
                ToolMessage(
                    content=json.dumps(tool_result),
                    name=tool["name"],
                    tool_call_id=tool["id"],
                )
            )
        return {"messages": outputs}

rag = LLM_agent(llm_model, tools, user_infor="thao", session_id="1234")

In [44]:
while (1):
    query = input("Enter your query: ")
    if query == "q": 
        break
    rag.stream_run(query)

In [45]:
human_command = Command(
    resume={
        "name": "LangGraph",
        "birthday": "Jan 17, 2024",
    },
)
events = rag.graph.stream(human_command, rag.configs, stream_mode="values")
for event in events:
    if "messages" in event:
        event["messages"][-1].pretty_print()

================================ Human Message =================================

Can you look up when LangGraph was released? When you have the answer, use the human_assistance tool for review.


AssertionError: 

In [46]:
# rag.human_command("We, the experts are here to help! We'd recommend you check out LangGraph to build your agent."
#     " It's much more reliable and extensible than simple autonomous agents.")

rag.print_history()

================================ Human Message =================================

Can you look up when LangGraph was released? When you have the answer, use the human_assistance tool for review.


# code web

In [24]:
from typing import Annotated

from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


class State(TypedDict):
    messages: Annotated[list, add_messages]
    name: str
    birthday: str

from langchain_core.messages import ToolMessage
from langchain_core.tools import InjectedToolCallId, tool

from langgraph.types import Command, interrupt
@tool
def human_assistance(
    name: str, birthday: str, tool_call_id: Annotated[str, InjectedToolCallId]
) -> str:
    """Request assistance from a human."""
    human_response = interrupt(
        {
            "question": "Is this correct?",
            "name": name,
            "birthday": birthday,
        },
    )
    # If the information is correct, update the state as-is.
    if human_response.get("correct", "").lower().startswith("y"):
        verified_name = name
        verified_birthday = birthday
        response = "Correct"
    # Otherwise, receive information from the human reviewer.
    else:
        verified_name = human_response.get("name", name)
        verified_birthday = human_response.get("birthday", birthday)
        response = f"Made a correction: {human_response}"

    # This time we explicitly update the state with a ToolMessage inside
    # the tool.
    state_update = {
        "name": verified_name,
        "birthday": verified_birthday,
        "messages": [ToolMessage(response, tool_call_id=tool_call_id)],
    }
    # We return a Command object in the tool to update our state.
    return Command(update=state_update)

from langchain_anthropic import ChatAnthropic
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode, tools_condition


tools = [search, human_assistance]
llm = llm_model
llm_with_tools = llm.bind_tools(tools)


def chatbot(state: State):
    message = llm_with_tools.invoke(state["messages"])
    assert len(message.tool_calls) <= 1
    return {"messages": [message]}


graph_builder = StateGraph(State)
graph_builder.add_node("chatbot", chatbot)

tool_node = ToolNode(tools=tools)
graph_builder.add_node("tools", tool_node)

graph_builder.add_conditional_edges(
    "chatbot",
    tools_condition,
)
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)

In [ ]:
user_input = (
    "Can you look up when LangGraph was released? "
    "When you have the answer, use the human_assistance tool for review."
)
configs = {"configurable": {"thread_id": "1"}}

events = graph.invoke(
    {"messages": [{"role": "user", "content": user_input}]},
    configs,
    stream_mode="values",
)
# for event in events:
#     if "messages" in event:
events["messages"][-1].pretty_print()